In [37]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from transformers import AutoTokenizer
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from collections import Counter
import numpy as np
import random

In [38]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

BATCH_SIZE = 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [39]:
dataset = load_dataset("Roblox/RobloxGuard-Eval")
print(dataset)

full = dataset["test"]

DatasetDict({
    test: Dataset({
        features: ['prompt', 'response', 'violation', 'category'],
        num_rows: 2873
    })
})


In [40]:
splits = full.train_test_split(test_size=0.1, seed=SEED)
train_data = splits["train"]
eval_data = splits["test"]
print("Train count: ", len(train_data))
print("Eval count: ", len(eval_data))

Train count:  2585
Eval count:  288


In [41]:
categories = sorted(set(train_data["category"]))
NUM_CLASSES = len(categories)
cat2idx = {c: i for i, c in enumerate(categories)}
idx2cat = {i: c for c, i in cat2idx.items()}
print("Total Categories: ", len(categories))
counts = Counter(train_data["category"])

for i, c in idx2cat.items():
    print("[", i, "]", c, ": ", str(counts[c]))

Total Categories:  24
[ 0 ]  :  1
[ 1 ] Cheating and Scams :  5
[ 2 ] Child Exploitation :  10
[ 3 ] Directing Users Off Platform :  63
[ 4 ] Discrimination, Slurs, and Hate Speech :  79
[ 5 ] Expanded Policies for Suitability :  8
[ 6 ] Illegal and Regulated Goods and Activities :  116
[ 7 ] Independent Advertisement Publishing :  4
[ 8 ] Intellectual Property Violations :  1
[ 9 ] Misusing Roblox Systems :  4
[ 10 ] None :  1773
[ 11 ] Paid Random Items :  2
[ 12 ] Political Figures and Entities :  74
[ 13 ] Profanity :  46
[ 14 ] Prohibited Advertising Practices and Content :  3
[ 15 ] Real-World Sensitive Events :  78
[ 16 ] Romantic and Sexual Content :  90
[ 17 ] Sharing Personal Information :  51
[ 18 ] Soliciting Donations :  5
[ 19 ] Spam :  11
[ 20 ] Suicide, Self Injury, and Harmful Behavior :  20
[ 21 ] Terrorism and Violent Extremism :  82
[ 22 ] Threats, Bullying, and Harassment :  32
[ 23 ] Violent Content and Gore :  27


In [42]:
tokenizer  = AutoTokenizer.from_pretrained("bert-base-uncased")
VOCAB_SIZE = tokenizer.vocab_size

In [43]:
def preprocess(data, tokenizer, max_len, cat2idx):
    input_ids_list = []
    attention_masks_list = []
    labels_list = []

    for row in data:
        text = f"{row['prompt']} [SEP] {row['response']}"
        encoder = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=max_len,
            return_tensors="pt",
        )

        label = cat2idx[row["category"]]
        input_ids_list.append(encoder["input_ids"].squeeze(0))
        attention_masks_list.append(encoder["attention_mask"].squeeze(0))
        labels_list.append(label)

    return {
        "input_ids": torch.stack(input_ids_list),
        "attention_mask": torch.stack(attention_masks_list),
        "labels": torch.tensor(
          labels_list,
          dtype=torch.long
        ),
    }

train_tensors = preprocess(train_data, tokenizer, 128, cat2idx)
eval_tensors  = preprocess(eval_data, tokenizer, 128, cat2idx)

In [44]:
from transformers import AutoModel

class BiLSTMClassifier(nn.Module):
    def __init__(self, hidden_dim, num_layers, num_classes, dropout):
        super().__init__()
        self.bert = AutoModel.from_pretrained("bert-base-uncased")
        for param in self.bert.parameters():
            param.requires_grad = False
        self.lstm = nn.LSTM(
            768, hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=True,
        )
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes),
        )

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            emb = self.bert(input_ids=input_ids,
                            attention_mask=attention_mask).last_hidden_state
        _, (h, _) = self.lstm(emb)
        out = torch.cat([h[-2], h[-1]], dim=1)
        return self.fc(self.drop(out))

model = BiLSTMClassifier(
    hidden_dim  = 256,
    num_layers  = 2,
    num_classes = NUM_CLASSES,
    dropout     = 0.3,
).to(DEVICE)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [45]:
cat_counts = np.array([
    sum(1 for x in train_data["category"] if x == c) for c in categories
], dtype=np.float32)

cat_counts = np.array(cat_counts, dtype=np.float32)

weights = torch.tensor(1.0 / cat_counts)
weights = (weights / weights.sum()).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=3
)

In [49]:
def train_epoch(model, tensors, criterion, optimizer, device):
    model.train()

    input_ids = tensors["input_ids"]
    attention_mask = tensors["attention_mask"]
    labels = tensors["labels"]

    total_loss = correct = total = 0

    for i in range(0, len(labels), BATCH_SIZE):
        ids  = input_ids[i:i+BATCH_SIZE].to(device)
        mask = attention_mask[i:i+BATCH_SIZE].to(device)
        lbls = labels[i:i+BATCH_SIZE].to(device)

        optimizer.zero_grad()
        logits = model(ids, mask)
        loss = criterion(logits, lbls)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * ids.size(0)
        correct += (logits.argmax(1) == lbls).sum().item()
        total += ids.size(0)

    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, tensors, criterion, device):
    model.eval()

    input_ids = tensors["input_ids"]
    attention_mask = tensors["attention_mask"]
    labels = tensors["labels"]

    total_loss = 0
    preds, targets = [], []

    for i in range(0, len(labels), BATCH_SIZE):
        ids  = input_ids[i:i+BATCH_SIZE].to(device)
        mask = attention_mask[i:i+BATCH_SIZE].to(device)
        lbls = labels[i:i+BATCH_SIZE].to(device)

        logits = model(ids, mask)
        loss = criterion(logits, lbls)

        total_loss += loss.item() * ids.size(0)
        preds.extend(logits.argmax(1).cpu().tolist())
        targets.extend(lbls.cpu().tolist())

    f1 = f1_score(targets, preds, average="macro", zero_division=0)
    acc = sum(p == t for p, t in zip(preds, targets)) / len(targets)

    return total_loss / len(labels), acc, f1, preds, targets

In [51]:
history    = {k: [] for k in ["tr_loss", "tr_acc", "ev_loss", "ev_acc", "ev_f1"]}
best_f1    = 0.0
best_state = None

for epoch in range(1, 20 + 1):
    tr_loss, tr_acc = train_epoch(model, train_tensors, criterion, optimizer, DEVICE)
    ev_loss, ev_acc, ev_f1, _, _ = evaluate(model, eval_tensors, criterion, DEVICE)

    scheduler.step(ev_f1)

    for k, v in zip(history, [tr_loss, tr_acc, ev_loss, ev_acc, ev_f1]):
        history[k].append(v)

    if ev_f1 > best_f1:
        best_f1 = ev_f1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f"best model saved")

    print(f"Epoch {epoch}/{20}  "
          f"train loss={tr_loss:.4f} acc={tr_acc:.4f}  |  "
          f"eval loss={ev_loss:.4f} acc={ev_acc:.4f} f1={ev_f1:.4f}")


best model saved
Epoch 1/20  train loss=0.0978 acc=0.8975  |  eval loss=4.7243 acc=0.7569 f1=0.3569
best model saved
Epoch 2/20  train loss=0.0696 acc=0.9207  |  eval loss=4.7148 acc=0.8021 f1=0.3743
Epoch 3/20  train loss=0.0601 acc=0.9273  |  eval loss=5.1553 acc=0.7882 f1=0.3701
best model saved
Epoch 4/20  train loss=0.0515 acc=0.9493  |  eval loss=4.4910 acc=0.7986 f1=0.4125
Epoch 5/20  train loss=0.0315 acc=0.9427  |  eval loss=4.7917 acc=0.8090 f1=0.4068
Epoch 6/20  train loss=0.0325 acc=0.9594  |  eval loss=5.2812 acc=0.8021 f1=0.3760
Epoch 7/20  train loss=0.0272 acc=0.9598  |  eval loss=5.0587 acc=0.8056 f1=0.4012
Epoch 8/20  train loss=0.0402 acc=0.9663  |  eval loss=4.5660 acc=0.8056 f1=0.3999
best model saved
Epoch 9/20  train loss=0.0160 acc=0.9718  |  eval loss=5.2762 acc=0.8125 f1=0.4248
Epoch 10/20  train loss=0.0117 acc=0.9799  |  eval loss=5.0577 acc=0.8056 f1=0.4142
Epoch 11/20  train loss=0.0099 acc=0.9857  |  eval loss=5.3340 acc=0.8125 f1=0.4120
best model saved


In [52]:
model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
_, acc, f1, preds, targets = evaluate(model, eval_tensors, criterion, DEVICE)
print(f"Accuracy: {acc:.4f}  |  Macro F1: {f1:.4f}\n")
print(classification_report(targets, preds, zero_division=0))

Accuracy: 0.8194  |  Macro F1: 0.4364

              precision    recall  f1-score   support

           2       0.00      0.00      0.00         4
           3       0.20      0.25      0.22         4
           4       0.67      0.67      0.67         6
           6       0.60      0.75      0.67         8
           9       0.00      0.00      0.00         1
          10       0.92      0.92      0.92       207
          12       0.67      0.86      0.75         7
          13       1.00      0.25      0.40         4
          15       0.81      0.68      0.74        19
          16       0.44      0.44      0.44         9
          17       0.75      0.75      0.75         4
          19       1.00      0.50      0.67         2
          20       0.00      0.00      0.00         3
          21       0.75      0.75      0.75         8
          22       0.00      0.00      0.00         2
          23       0.00      0.00      0.00         0

    accuracy                           0.